# Two-Stage Inference Pipeline Demonstration

This notebook chains the Stage 1 Tumor Classifier and the Stage 2 Severity Classifier to perform complete inference on single brain MRI scans.

In [ ]:
import sys
sys.path.append('..')
from full_pipeline import load_models, full_pipeline
from config        import *
import matplotlib.pyplot as plt
from PIL            import Image

print(f"Pipeline imported | Device: {DEVICE}")

In [ ]:
stage1_model, stage2_model = load_models()
print("Stage 1 and Stage 2 models loaded successfully!")

In [ ]:
import os
from pathlib import Path

test_images = {}
for cls in TUMOR_CLASSES:
    cls_dir = Path('../data/tumor_type/Testing') / cls
    if cls_dir.exists():
        imgs = list(cls_dir.glob('*.jpg')) + \
               list(cls_dir.glob('*.jpeg')) + \
               list(cls_dir.glob('*.png'))
        if imgs:
            test_images[cls] = imgs[0]

print("Found sample test images:")
for cls, path in test_images.items():
    print(f"  {cls:12s}: {path}")

In [ ]:
# Run pipeline on a tumor sample
sample_class = 'meningioma' if 'meningioma' in test_images else list(test_images.keys())[0]
sample_path  = test_images[sample_class]

print(f"Running pipeline on sample of type '{sample_class}' at: {sample_path}")
result = full_pipeline(sample_path, stage1_model, stage2_model)

# Display result
img = Image.open(sample_path)
plt.imshow(img)
plt.title(f"Predicted Type: {result['type']}\nSubclass: {result.get('severity', '—')} | Risk: {result['risk']}")
plt.axis('off')
plt.show()

print("Pipeline Output:")
for k, v in result.items():
    print(f"  {k:10s}: {v}")

In [ ]:
# Run pipeline on a 'notumor' sample
if 'notumor' in test_images:
    sample_path = test_images['notumor']
    print(f"Running pipeline on 'notumor' sample at: {sample_path}")
    result = full_pipeline(sample_path, stage1_model, stage2_model)
    
    # Display result
    img = Image.open(sample_path)
    plt.imshow(img)
    plt.title(f"Predicted Type: {result['type']} | Risk: {result['risk']}")
    plt.axis('off')
    plt.show()
    
    print("Pipeline Output:")
    for k, v in result.items():
        print(f"  {k:10s}: {v}")